# 5. Ground truth: the exact contributing-tissue field

For each ground-truth recording (clean, catheter centres in `groundTruth.centresYmm`, both gaps and
conditions) the reciprocity lead field gives every element's signed share of the signal energy in the
window. Shares sum to 1; nothing is cut off; negative shares (cancellation) are kept. Fields are summed
into 0.5 mm voxels (`groundTruth/contributionField.npz`).

Checks, enforced: shares sum to 1, the field reproduces the measured clean electrogram, and its scale
equals the calibration gain.

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_CONFIG / EFOV_OUT / EFOV_NP ----
CONFIG = os.environ.get("EFOV_CONFIG", str(REPOSITORY / "config" / "final_model.json"))
OUT    = os.environ.get("EFOV_OUT",    str(REPOSITORY / "runs" / "v3_reference"))
NP     = int(os.environ.get("EFOV_NP", "24"))      # MPI processes for openCARP
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.config import loadConfig
from efov.pipeline import Run

config = loadConfig(CONFIG)
run = Run(config, OUT, processCount=NP)
print(f"{config['modelName']} -> {run.out}")
run.status()

In [ ]:
checks = run.runStage("groundTruth")
checks

In [ ]:
summary = pd.read_csv(run.out / "groundTruth" / "summary.csv")
summary

In [ ]:
from efov.report import fieldFigure
path = fieldFigure(run, run.folder("figures"))
if path: display(Image(str(path)))

In [ ]:
import matplotlib.pyplot as plt
field = np.load(run.out / "groundTruth" / "contributionField.npz")
step = float(field["curveStepMm"])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for k, recording in enumerate(field["recordingId"]):
    if ":0:" in recording and "intact" in recording:
        axes[0].plot(np.arange(field["lateralCumulativeShare"].shape[1]) * step, field["lateralCumulativeShare"][k], lw=0.8)
        axes[1].plot(np.arange(field["depthCumulativeShare"].shape[1]) * step, field["depthCumulativeShare"][k], lw=0.8)
axes[0].set(xlabel="lateral distance from anchor (mm)", ylabel="cumulative share", title="intact recordings")
axes[1].set(xlabel="depth below the catheter-facing surface (mm)", title="descriptive curves, not a radius")
plt.show()